# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

High-Performance Cloud GPU inference pipeline (`large-v3-turbo` + `ECAPA-TDNN` + `Gemini 2.5 Flash`).

---
### Quick Start:
1. In the menu bar above, click **Runtime → Run all** (or press `Ctrl + F9`).
2. Use the interactive web application embedded below or click the public live link!

In [ ]:
# 1. Install GPU Acceleration Libraries
!pip install -q faster-whisper speechbrain gradio torchaudio soundfile scipy


In [ ]:
# 2. Launch Full-Fidelity VoiceDiary Web Platform (Exact Desktop UI/UX)
import os, time, tempfile, json, urllib.request
import numpy as np
import soundfile as sf
import gradio as gr
import torch
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# Hardware Acceleration Telemetry
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Multi-Core (AVX2)'
compute_dtype = 'float16' if torch.cuda.is_available() else 'int8'
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Hardware Engine Online: {gpu_name} ({compute_dtype})')

# Global Model Cache
loaded_models = {}

def get_whisper_model(model_name='base'):
    if model_name not in loaded_models:
        print(f'Loading Whisper {model_name} on {device_type}...')
        loaded_models[model_name] = WhisperModel(model_name, device=device_type, compute_type=compute_dtype)
    return loaded_models[model_name]

# Pre-load Base for instantaneous startup
whisper_model = get_whisper_model('base')

# Load ECAPA-TDNN Diarizer on GPU
with tempfile.TemporaryDirectory() as tmp:
    embedder = EncoderClassifier.from_hparams(
        source='speechbrain/spkrec-ecapa-voxceleb',
        savedir=tmp,
        run_opts={'device': device_type}
    )

# Full Obsidian & Amber Gold Desktop CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

:root {
    --bg-primary: #090D16 !important;
    --bg-card: #0F172A !important;
    --border-color: #1E293B !important;
    --accent-gold: #F59E0B !important;
}

body, .gradio-container {
    background-color: #090D16 !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    color: #F8FAFC !important;
    max-width: 1400px !important;
    margin: 0 auto !important;
}

/* Header */
.vd-header {
    background: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 16px;
    padding: 16px 24px;
    margin-bottom: 18px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    box-shadow: 0 10px 30px -10px rgba(0,0,0,0.6);
}

.brand-wrapper {
    display: flex;
    align-items: center;
    gap: 14px;
}

.brand-logo-badge {
    width: 44px;
    height: 44px;
    background: linear-gradient(135deg, #F59E0B, #D97706);
    border-radius: 12px;
    display: flex;
    align-items: center;
    justify-content: center;
    color: #090D16;
    box-shadow: 0 0 20px rgba(245, 158, 11, 0.35);
}

.brand-title {
    font-size: 20px;
    font-weight: 800;
    color: #FFFFFF;
    letter-spacing: -0.02em;
    margin: 0;
}

.brand-sub {
    font-size: 11px;
    color: #94A3B8;
    font-weight: 600;
}

.hw-badge-pill {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 6px 16px;
    border-radius: 9999px;
    background: rgba(16, 185, 129, 0.1);
    border: 1px solid rgba(16, 185, 129, 0.3);
    color: #34D399;
    font-size: 12px;
    font-weight: 700;
    font-family: 'Fira Code', monospace;
}

/* Panels */
.vd-card {
    background: #0F172A !important;
    border: 1px solid #1E293B !important;
    border-radius: 16px !important;
    padding: 20px !important;
}

.btn-gold-primary {
    background: linear-gradient(135deg, #F59E0B, #D97706) !important;
    color: #000000 !important;
    font-weight: 800 !important;
    font-size: 15px !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 14px 24px !important;
    box-shadow: 0 0 20px rgba(245, 158, 11, 0.3) !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
}
.btn-gold-primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 0 30px rgba(245, 158, 11, 0.5) !important;
}

.speaker-item {
    background: rgba(255, 255, 255, 0.03);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 12px;
    padding: 12px 14px;
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 10px;
}

.spk-avatar {
    width: 36px;
    height: 36px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    font-size: 14px;
    color: #FFFFFF;
}

.transcript-panel {
    background: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 16px;
    padding: 24px;
    min-height: 520px;
    max-height: 650px;
    overflow-y: auto;
}
"""

def run_pipeline(audio_path, model_choice, lang_choice, sim_threshold):
    if not audio_path or not os.path.exists(audio_path):
        return "<div style='color:#EF4444;padding:30px;text-align:center;'>Please record audio or upload a classroom lecture file.</div>", "<div style='color:#64748B;padding:20px;text-align:center;'>No speakers active</div>", None, None, None
    
    t0 = time.time()
    
    # Load Model Dynamically
    model_key_map = {
        'Large-v3-Turbo (809M) - SOTA Accuracy': 'large-v3-turbo',
        'Whisper Base (74M) - Recommended Fast': 'base',
        'Whisper Tiny (39M) - Ultralight': 'tiny',
        'Whisper Small (244M) - High Accuracy': 'small',
        'Whisper Medium (769M) - Deep Precision': 'medium',
        'Distil-Whisper (756M) - English Fast': 'distil-large-v3'
    }
    selected_model_key = model_key_map.get(model_choice, 'base')
    model = get_whisper_model(selected_model_key)
    
    # Read and resample audio
    data, sr = sf.read(audio_path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    data = data.astype(np.float32)
    
    if sr != 16000:
        from scipy.signal import resample_poly
        gcd = int(np.gcd(16000, sr))
        data = resample_poly(data, 16000 // gcd, sr // gcd).astype(np.float32)
        
    duration = len(data) / 16000.0
    
    lang_map = {
        'Bilingual (Auto Urdu + English)': None,
        'Urdu Script (اردو)': 'ur',
        'English Only': 'en'
    }
    target_lang = lang_map.get(lang_choice, None)
    
    # Transcribe with Whisper
    segments, info = model.transcribe(
        data,
        beam_size=3,
        language=target_lang,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )
    
    speaker_profiles = {}
    speaker_colors = ['#6366F1', '#10B981', '#F59E0B', '#EF4444', '#EC4899', '#8B5CF6', '#06B6D4']
    next_speaker_id = 1
    
    transcript_html = []
    plain_entries = []
    
    thresh = float(sim_threshold) / 100.0
    
    for seg in segments:
        text = seg.text.strip()
        if not text:
            continue
            
        start_s = seg.start
        end_s = seg.end
        seg_audio = data[int(start_s * 16000): int(end_s * 16000)]
        
        spk_id = 1
        if len(seg_audio) >= 8000:
            try:
                with torch.inference_mode():
                    waveform = torch.from_numpy(seg_audio).float().unsqueeze(0).to(device_type)
                    emb = embedder.encode_batch(waveform).squeeze().detach().cpu().numpy()
                    emb_norm = emb / (np.linalg.norm(emb) or 1.0)
                    
                best_id = None
                best_sim = -1.0
                for s_id, embs in speaker_profiles.items():
                    sims = [float(np.dot(emb_norm, e)) for e in embs]
                    max_s = max(sims) if sims else 0
                    if max_s > best_sim:
                        best_sim = max_s
                        best_id = s_id
                        
                if best_id and best_sim >= thresh:
                    spk_id = best_id
                    if len(speaker_profiles[spk_id]) < 50:
                        speaker_profiles[spk_id].append(emb_norm)
                else:
                    spk_id = next_speaker_id
                    speaker_profiles[spk_id] = [emb_norm]
                    next_speaker_id += 1
            except Exception:
                pass
                
        spk_color = speaker_colors[(spk_id - 1) % len(speaker_colors)]
        time_str = f"{int(start_s // 60):02d}:{int(start_s % 60):02d}"
        
        is_urdu = any('؀' <= char <= 'ۿ' for char in text)
        rtl_style = "direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; font-size: 16px; line-height: 1.8;" if is_urdu else "font-size: 14px; line-height: 1.6;"
        
        node = f"""
        <div style="margin-bottom: 14px; padding: 14px 18px; border-left: 4px solid {spk_color}; background: rgba(255,255,255,0.02); border-radius: 10px; border: 1px solid rgba(255,255,255,0.04);">
            <div style="display:flex; align-items:center; gap:8px; margin-bottom:6px;">
                <span style="display:inline-block; width:8px; height:8px; border-radius:50%; background:{spk_color};"></span>
                <strong style="color:{spk_color}; font-size:13px;">Speaker {spk_id}</strong>
                <span style="color:#64748B; font-size:11px; font-family:'Fira Code', monospace;">[{time_str}]</span>
            </div>
            <div style="color:#F1F5F9; {rtl_style}">{text}</div>
        </div>
        """
        transcript_html.append(node)
        plain_entries.append(f"[{time_str}] Speaker {spk_id}: {text}")
        
    elapsed = time.time() - t0
    
    # Render Sidebar Speaker Profiles
    sidebar_html = []
    for s_id, embs in speaker_profiles.items():
        c = speaker_colors[(s_id - 1) % len(speaker_colors)]
        sidebar_html.append(f"""
        <div class="speaker-item">
            <div class="spk-avatar" style="background:{c};">S{s_id}</div>
            <div>
                <div style="font-weight:700; font-size:13px; color:#FFFFFF;">Speaker {s_id}</div>
                <div style="font-size:11px; color:#94A3B8;">{len(embs)} voice prints</div>
            </div>
        </div>
        """)
    if not sidebar_html:
        sidebar_html.append("<div style='color:#64748B;padding:16px;text-align:center;'>No speakers enrolled</div>")
        
    full_html = "".join(transcript_html)
    full_html += f"""
    <div style="margin-top:20px; padding-top:14px; border-top:1px solid #1E293B; font-size:12px; color:#94A3B8; display:flex; justify-content:space-between;">
        <span>Hardware Engine: {gpu_name}</span>
        <span>Processed {duration:.1f}s in {elapsed:.2f}s ({(duration/max(0.01, elapsed)):.1f}x real-time)</span>
    </div>
    """
    
    # Save files
    with tempfile.NamedTemporaryFile(mode='w', suffix='.md', delete=False, encoding='utf-8') as f:
        f.write(f"# VoiceDiary Lecture Notes\n\n" + "\n\n".join(plain_entries))
        md_file = f.name
        
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
        f.write("\n".join(plain_entries))
        txt_file = f.name
        
    return full_html, "".join(sidebar_html), md_file, txt_file, "\n".join(plain_entries)

def generate_gemini_summary(transcript_text, gemini_api_key):
    if not transcript_text or not transcript_text.strip():
        return "No transcript content available to summarize."
        
    api_key = gemini_api_key.strip() if gemini_api_key else os.environ.get('GEMINI_API_KEY', '')
    if not api_key:
        return "Please paste your Gemini API Key in the left settings panel to generate structured study summaries."
        
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    payload = {
        "contents": [{
            "parts": [{
                "text": f"""You are VoiceDiary AI, an expert academic classroom study summarizer.
Analyze the following diarized lecture transcript and produce a structured Markdown study summary:
1. Executive Lecture Overview
2. Core Technical Concepts Discussed
3. Key Takeaways & Exam Points
4. Q&A / Discussion Highlights

Transcript:
{transcript_text}"""
            }]
        }]
    }
    
    try:
        req = urllib.request.Request(
            url,
            data=json.dumps(payload).encode('utf-8'),
            headers={'Content-Type': 'application/json'}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode('utf-8'))
            return data['candidates'][0]['content']['parts'][0]['text']
    except Exception as e:
        return f"Gemini API Error: {e}"

with gr.Blocks(title="VoiceDiary AI — Classroom Lecture Platform", css=custom_css, theme=gr.themes.Default(primary_hue="amber", neutral_hue="slate")) as demo:
    transcript_state = gr.State("")
    
    gr.HTML(f"""
    <div class="vd-header">
        <div class="brand-wrapper">
            <div class="brand-logo-badge">
                <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"></path><path d="M19 10v2a7 7 0 0 1-14 0v-2"></path><line x1="12" y1="19" x2="12" y2="23"></line><line x1="8" y1="23" x2="16" y2="23"></line></svg>
            </div>
            <div>
                <div class="brand-title">VoiceDiary</div>
                <div class="brand-sub">Bilingual Classroom AI & Speaker Diarization Engine © Abdul Sarim Khan</div>
            </div>
        </div>
        <div style="display:flex; align-items:center; gap:16px;">
            <div class="hw-badge-pill">GPU: {gpu_name} (Tensor Cores FP16)</div>
        </div>
    </div>
    """)
    
    with gr.Row():
        # LEFT COLUMN: Speakers & Controls (Matches Desktop Sidebar)
        with gr.Column(scale=3):
            with gr.Group(elem_classes=["vd-card"]):
                gr.HTML("<div style='font-size:13px; font-weight:800; color:#F59E0B; margin-bottom:12px; letter-spacing:0.05em;'>SPEAKERS & PROFILES</div>")
                sidebar_out = gr.HTML(value="<div style='color:#64748B;padding:16px;text-align:center;'>Start transcription to enroll speaker voiceprints</div>")
                
            with gr.Group(elem_classes=["vd-card"], visible=True):
                gr.HTML("<div style='font-size:13px; font-weight:800; color:#F59E0B; margin-bottom:12px; letter-spacing:0.05em;'>AI MODEL & ENGINE SETTINGS</div>")
                model_dropdown = gr.Dropdown(
                    choices=[
                        'Large-v3-Turbo (809M) - SOTA Accuracy',
                        'Whisper Base (74M) - Recommended Fast',
                        'Whisper Tiny (39M) - Ultralight',
                        'Whisper Small (244M) - High Accuracy',
                        'Whisper Medium (769M) - Deep Precision',
                        'Distil-Whisper (756M) - English Fast'
                    ],
                    value='Whisper Base (74M) - Recommended Fast',
                    label='Active Whisper Model'
                )
                lang_dropdown = gr.Dropdown(
                    choices=[
                        'Bilingual (Auto Urdu + English)',
                        'Urdu Script (اردو)',
                        'English Only'
                    ],
                    value='Bilingual (Auto Urdu + English)',
                    label='Language Mode'
                )
                thresh_slider = gr.Slider(minimum=20, maximum=70, value=32, step=1, label='Diarization Sensitivity (Cosine Threshold %)')
                gemini_key_in = gr.Textbox(placeholder='Paste Gemini API Key (Optional)...', type='password', label='Gemini AI Key (BYOK)')

        # RIGHT COLUMN: Main Audio Input & Real-Time Classroom Transcript
        with gr.Column(scale=7):
            with gr.Group(elem_classes=["vd-card"]):
                with gr.Tabs():
                    with gr.TabItem("Live Classroom Microphone"):
                        audio_mic = gr.Audio(sources=["microphone"], type="filepath", label="Capture Classroom Speech")
                    with gr.TabItem("Upload Lecture Audio File"):
                        audio_file = gr.Audio(sources=["upload"], type="filepath", label="Upload Audio File (.wav, .mp3, .m4a, .flac)")
                
                transcribe_btn = gr.Button("Transcribe & Diarize Lecture", elem_classes=["btn-gold-primary"])
            
            with gr.Group(elem_classes=["vd-card"]):
                gr.HTML("<div style='font-size:14px; font-weight:700; color:#FFFFFF; margin-bottom:10px;'>Live Diarized Classroom Lecture Notes</div>")
                transcript_display = gr.HTML(
                    value="<div style='display:flex; flex-direction:column; align-items:center; justify-content:center; height:320px; color:#64748B; text-align:center;'><div>Lecture transcript will stream here.<br/>Click the gold button above to begin.</div></div>",
                    elem_classes=["transcript-panel"]
                )
            
            # Export & Gemini AI Summary
            with gr.Group(elem_classes=["vd-card"]):
                gr.HTML("<div style='font-size:13px; font-weight:800; color:#F59E0B; margin-bottom:10px;'>EXPORT & AI LECTURE STUDY NOTES</div>")
                with gr.Row():
                    d_md = gr.File(label="Markdown Notes (.md)")
                    d_txt = gr.File(label="Plain Text (.txt)")
                
                ai_sum_btn = gr.Button("Generate AI Lecture Summary (Gemini 2.5 Flash)")
                summary_out = gr.Markdown(value="*AI summary and study notes will appear here after clicking above...*")

    # Wire event handlers
    transcribe_btn.click(
        fn=lambda m, f, mod, lang, th: run_pipeline(m if m else f, mod, lang, th),
        inputs=[audio_mic, audio_file, model_dropdown, lang_dropdown, thresh_slider],
        outputs=[transcript_display, sidebar_out, d_md, d_txt, transcript_state]
    )
    
    ai_sum_btn.click(
        fn=generate_gemini_summary,
        inputs=[transcript_state, gemini_key_in],
        outputs=[summary_out]
    )

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
